# Notebook 04: Out-of-Sample Validation

This notebook performs out-of-sample validation of the continuous HMM (CHMM, no jumps).
The model is trained on in-sample data and evaluated on a held-out period.

### Validation Strategy
1. Train CHMM on IS data via Baum-Welch
2. Simulate OoS-length paths from the learned model
3. Compare simulated paths against observed OoS returns

### Metrics
1. **KS pass rate** (%) -- Kolmogorov-Smirnov test at α=0.05
2. **Excess kurtosis** -- simulated vs observed
3. **ACF-MAE** -- mean absolute error of ACF(|Gₜ|) over lags 1-252
4. **Wasserstein-1 distance** -- mean earth-mover distance
5. **Hellinger distance** -- histogram overlap distance

### Outputs
- **Figure 4**: Three-panel OoS validation (KS p-values, density fan chart, ACF)
- **OoS vs IS metric comparison**

## Setup

In [ ]:
include("../Include.jl");

## Configuration

In [ ]:
# --- TUNABLE PARAMETERS ---
ticker = "SPY";
K = 6;
MAX_ITER = 60;
N_PATHS = 1000;
L = 252;
risk_free_rate = 0.0;
Δt = 1/252;

## Load In-Sample and Out-of-Sample Data

In [ ]:
# --- IN-SAMPLE DATA ---
train_dataset = MyPortfolioDataSet() |> x -> x["dataset"];
maximum_number_trading_days = nrow(train_dataset["AAPL"]);

dataset = Dict{String,DataFrame}();
for (t, data) ∈ train_dataset
    if nrow(data) == maximum_number_trading_days
        dataset[t] = data;
    end
end

list_of_all_tickers = keys(dataset) |> collect |> sort;
all_firms_R = log_growth_matrix(dataset, list_of_all_tickers; Δt=Δt, risk_free_rate=risk_free_rate);
ticker_idx = findfirst(x -> x == ticker, list_of_all_tickers);
R_is = all_firms_R[:, ticker_idx];
n_steps_is = length(R_is);

println("IS data: $(n_steps_is) observations for $ticker")

In [ ]:
# --- OUT-OF-SAMPLE DATA ---
oos_dataset_raw = MyOutOfSamplePortfolioDataSet() |> x -> x["dataset"];
R_oos = log_growth_matrix(oos_dataset_raw, ticker; Δt=Δt, risk_free_rate=risk_free_rate);
n_steps_oos = length(R_oos);

println("OoS data: $(n_steps_oos) observations for $ticker")

## Train CHMM on In-Sample Data

In [ ]:
# --- TRAIN MODEL ---
model = build(MyContinuousHiddenMarkovModel, (
    observations = R_is,
    number_of_states = K,
    max_iter = MAX_ITER
));

println("Training complete: $(length(model.log_likelihood_history)) iterations, K=$(K)")

In [ ]:
# --- TRANSITION MATRIX AND STATIONARY DISTRIBUTION ---
T_matrix = zeros(K, K);
for i in 1:K
    T_matrix[i, :] = probs(model.transition[i]);
end

π_stationary = (T_matrix^1000)[1, :];
start_dist = Categorical(π_stationary);

println("Stationary distribution: ", round.(π_stationary, digits=4))
println("Sum: ", round(sum(π_stationary), digits=6))

## Simulate OoS-Length Paths

In [ ]:
# --- SIMULATE OoS-LENGTH PATHS (CHMM, no jumps) ---
decoded_oos = Array{Float64,2}(undef, n_steps_oos, N_PATHS);

for i in 1:N_PATHS
    s0 = rand(start_dist);
    states = model(s0, n_steps_oos);
    for j in 1:n_steps_oos
        decoded_oos[j, i] = rand(model.emission[states[j]]);
    end
end

println("OoS simulation complete: $(N_PATHS) paths x $(n_steps_oos) steps.")

## Simulate IS-Length Paths (for comparison)

In [ ]:
# --- SIMULATE IS-LENGTH PATHS (CHMM, no jumps) ---
decoded_is = Array{Float64,2}(undef, n_steps_is, N_PATHS);

for i in 1:N_PATHS
    s0 = rand(start_dist);
    states = model(s0, n_steps_is);
    for j in 1:n_steps_is
        decoded_is[j, i] = rand(model.emission[states[j]]);
    end
end

println("IS simulation complete: $(N_PATHS) paths x $(n_steps_is) steps.")

## Compute Validation Metrics

In [ ]:
# --- VALIDATION METRICS FUNCTION ---
function eval_metrics(observed::Vector{Float64}, sim_archive::Matrix{Float64}, L_val::Int)
    np = size(sim_archive, 2);
    n_o = length(observed);
    μ_o = mean(observed); σ_o = std(observed);
    kurt_obs_val = sum(((observed .- μ_o) ./ σ_o).^4) / n_o - 3.0;
    L_use = min(L_val, n_o - 1);
    acf_obs_val = autocor(abs.(observed), 1:L_use);

    ks_pass = 0; kurt_s = 0.0; acf_mae_s = 0.0; w1_s = 0.0; hell_s = 0.0;
    ks_pvals = Float64[];

    for i in 1:np
        sim = sim_archive[:, i];
        pval = pvalue(ApproximateTwoSampleKSTest(observed, sim));
        push!(ks_pvals, pval);
        if pval > 0.05; ks_pass += 1; end

        μ_s = mean(sim); σ_s = std(sim);
        kurt_s += sum(((sim .- μ_s) ./ σ_s).^4) / length(sim) - 3.0;

        acf_sim_val = autocor(abs.(sim), 1:L_use);
        acf_mae_s += mean(abs.(acf_obs_val .- acf_sim_val));

        # Wasserstein-1
        obs_s = sort(observed); sim_s = sort(sim);
        n_min = min(length(obs_s), length(sim_s));
        obs_q = [obs_s[max(1, round(Int, k*length(obs_s)/n_min))] for k in 1:n_min];
        sim_q = [sim_s[max(1, round(Int, k*length(sim_s)/n_min))] for k in 1:n_min];
        w1_s += mean(abs.(obs_q .- sim_q));

        # Hellinger
        lo = min(minimum(observed), minimum(sim)) - 10;
        hi = max(maximum(observed), maximum(sim)) + 10;
        edges = range(lo, hi, length=101);
        h_o = fit(Histogram, observed, edges).weights ./ n_o;
        h_s = fit(Histogram, sim, edges).weights ./ length(sim);
        hell_s += sqrt(sum((sqrt.(h_o) .- sqrt.(h_s)).^2)) / sqrt(2);
    end

    return (ks_rate=round(100*ks_pass/np, digits=1),
            kurtosis_obs=round(kurt_obs_val, digits=2),
            kurtosis_sim=round(kurt_s/np, digits=2),
            acf_mae=round(acf_mae_s/np, digits=4),
            wasserstein=round(w1_s/np, digits=3),
            hellinger=round(hell_s/np, digits=4),
            ks_pvals=ks_pvals)
end;

In [ ]:
# --- COMPUTE METRICS ---
m_oos = eval_metrics(R_oos, decoded_oos, L);
m_is = eval_metrics(R_is, decoded_is, L);

println("\n--- CHMM OoS Metrics ($(N_PATHS) paths) ---");
println("KS pass rate:    $(m_oos.ks_rate)%");
println("Excess kurtosis: $(m_oos.kurtosis_sim) (observed: $(m_oos.kurtosis_obs))");
println("ACF-MAE:         $(m_oos.acf_mae)");
println("Wasserstein-1:   $(m_oos.wasserstein)");
println("Hellinger:       $(m_oos.hellinger)");

println("\n--- CHMM IS Metrics ($(N_PATHS) paths) ---");
println("KS pass rate:    $(m_is.ks_rate)%");
println("Excess kurtosis: $(m_is.kurtosis_sim) (observed: $(m_is.kurtosis_obs))");
println("ACF-MAE:         $(m_is.acf_mae)");
println("Wasserstein-1:   $(m_is.wasserstein)");
println("Hellinger:       $(m_is.hellinger)");

## Figure 4: Out-of-Sample Validation

Three-panel figure:
- **(a)** KS p-value histogram with α=0.05 line
- **(b)** Density fan chart: 50 light simulated densities + bold observed OoS
- **(c)** OoS ACF(|Gₜ|) with mean and 10-90th percentile band

In [ ]:
# --- FIGURE 4a: KS p-value histogram ---
p4a = histogram(m_oos.ks_pvals, bins=50, normalize=true, alpha=0.6, color=:steelblue,
    label="CHMM (K=$K)", title="(a) OoS KS p-values (pass=$(m_oos.ks_rate)%)", titlefontsize=9,
    xlabel="p-value", ylabel="Density");
vline!(p4a, [0.05], lw=2, color=:red, ls=:dash, label="\u03b1 = 0.05");

# --- FIGURE 4b: Density Fan Chart ---
p4b = plot(title="(b) OoS Density Fan Chart", titlefontsize=9,
    xlabel="Excess Growth Rate", ylabel="Density");

for i in 1:min(50, N_PATHS)
    density!(p4b, decoded_oos[:, i], color=:steelblue, alpha=0.06, label="");
end
density!(p4b, R_oos, lw=3, color=:red, label="Observed OoS");

# --- FIGURE 4c: OoS ACF(|G_t|) ---
L_oos = min(L, n_steps_oos - 1);
τ = 1:L_oos;
acf_oos_obs = autocor(abs.(R_oos), τ);

n_acf_paths = min(200, N_PATHS);
acf_oos_archive = hcat([autocor(abs.(decoded_oos[:, i]), τ) for i in 1:n_acf_paths]...);
acf_oos_mean = mean(acf_oos_archive, dims=2)[:];
acf_oos_p10 = [quantile(acf_oos_archive[t, :], 0.10) for t in 1:L_oos];
acf_oos_p90 = [quantile(acf_oos_archive[t, :], 0.90) for t in 1:L_oos];

p4c = plot(τ, acf_oos_obs, lw=2, color=:red, ls=:dash, label="Observed OoS",
    title="(c) OoS ACF(|G\u209c|)", titlefontsize=9,
    xlabel="Lag (trading days)", ylabel="ACF(|G\u209c|)");
plot!(p4c, τ, acf_oos_mean, lw=2, color=:steelblue, label="CHMM mean");
plot!(p4c, τ, acf_oos_p10, fillrange=acf_oos_p90, alpha=0.2, color=:steelblue, label="10-90th pctl");

# --- COMBINE ---
fig4 = plot(p4a, p4b, p4c, layout=(1, 3), size=(1400, 400),
    plot_title="Figure 4: Out-of-Sample Validation \u2014 $(ticker) (CHMM, K=$(K))",
    plot_titlefontsize=12);
display(fig4)

savefig(fig4, joinpath(_PATH_TO_FIGURES, "Fig-4-OoS-Validation-$(ticker)-K$(K).svg"));
savefig(fig4, joinpath(_PATH_TO_FIGURES, "Fig-4-OoS-Validation-$(ticker)-K$(K).pdf"));

## OoS vs IS Metric Comparison

In [ ]:
# --- SUMMARY TABLE: IS vs OoS ---
println("\n" * "="^65);
println("  OoS vs IS Comparison \u2014 $(ticker), CHMM (K=$(K), $(N_PATHS) paths)");
println("="^65);
println("Metric              |   In-Sample   |  Out-of-Sample");
println("-"^65);
println("KS pass rate (%)    |  $(lpad(m_is.ks_rate, 10))  |  $(lpad(m_oos.ks_rate, 10))");
println("Kurt (observed)     |  $(lpad(m_is.kurtosis_obs, 10))  |  $(lpad(m_oos.kurtosis_obs, 10))");
println("Kurt (simulated)    |  $(lpad(m_is.kurtosis_sim, 10))  |  $(lpad(m_oos.kurtosis_sim, 10))");
println("ACF-MAE             |  $(lpad(m_is.acf_mae, 10))  |  $(lpad(m_oos.acf_mae, 10))");
println("Wasserstein-1       |  $(lpad(m_is.wasserstein, 10))  |  $(lpad(m_oos.wasserstein, 10))");
println("Hellinger           |  $(lpad(m_is.hellinger, 10))  |  $(lpad(m_oos.hellinger, 10))");
println("="^65);

## Disclaimer

This content is offered solely for research and educational purposes. It does not constitute financial advice.